# **Human In The Loop Middleware**

- interrupt_on: `dict[str, bool | InterruptOnConfig]` - Mapping of tool name to allowed actions. If a tool doesn't have an entry, it's auto-approved by default.
- description_prefix: `str` - The prefix to use when constructing action requests. Not used if a tool has a `description` in its `InterruptOnConfig`.


InterruptOnConfig defines configuration for an action requiring human in the loop. It carries following args:
- allowed_decisions: `DecisionType = Literal['approve', 'edit', 'reject']`
- description: `str`
- args_schema: `NotRequired[dict[str, Any]]` - JSON schema for the args associated with the action, if edits are allowed.

In [13]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model=chat_model,
    tools=[search_items, purchase_item],
    system_prompt=PROMPT,
    middleware=[
        ToolCallLimitMiddleware(
            tool_name="purchase_item",
            run_limit=1,  # Max 1 purchase per conversation turn
            thread_limit=2,  # Max 2 purchases total across all conversation turns
        ), 
        HumanInTheLoopMiddleware(
            interrupt_on={"purchase_item": True}
        )
    ],
)

In [14]:
response = agent.invoke({"messages": "find the headphones under 300$"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

find the headphones under 300$
================================== Ai Message ==================================
Tool Calls:
  search_items (fc_f9841677-bef5-4386-99da-5bcbe494be53)
 Call ID: fc_f9841677-bef5-4386-99da-5bcbe494be53
  Args:
    query: headphones under $300
================================= Tool Message =================================
Name: search_items

Found 3 items:
    - ID: hdpn-001, Sony WH-1000XM5, $299.99, rating 4.5/5
    - ID: hdpn-002, Bose QuietComfort 45, $279.00, rating 4.7/5
    - ID: hdpn-003, Apple AirPods Max, $549.00, rating 4.8/5
================================== Ai Message ==================================

Here are the headphones that fit your budget (under $300):

| ID | Model | Price | Rating |
|----|-------|-------|--------|
| **hdpn-002** | Bose QuietComfort 45 | **$279.00** | **4.7 / 5** |
| hdpn-001 | Sony WH‑1000XM5 | $299.99 | 4.5 / 5 |

The Bose QuietComfor

In [15]:
response = agent.invoke({"messages": "yes, please purchase Bose QuietComfort 45"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

yes, please purchase Bose QuietComfort 45
================================== Ai Message ==================================
Tool Calls:
  search_items (fc_f86431cc-eee8-4bd7-a34c-6351bf9b9d5d)
 Call ID: fc_f86431cc-eee8-4bd7-a34c-6351bf9b9d5d
  Args:
    query: Bose QuietComfort 45
================================= Tool Message =================================
Name: search_items

Found 3 items:
    - ID: hdpn-001, Sony WH-1000XM5, $299.99, rating 4.5/5
    - ID: hdpn-002, Bose QuietComfort 45, $279.00, rating 4.7/5
    - ID: hdpn-003, Apple AirPods Max, $549.00, rating 4.8/5
================================== Ai Message ==================================
Tool Calls:
  purchase_item (fc_c5c5a048-2f3c-4783-b439-0e95b3dd4dd6)
 Call ID: fc_c5c5a048-2f3c-4783-b439-0e95b3dd4dd6
  Args:
    item_id: hdpn-002


In [16]:
response

{'messages': [HumanMessage(content='yes, please purchase Bose QuietComfort\u202f45', additional_kwargs={}, response_metadata={}, id='2e90a381-801d-4d88-87a2-f427fc96734c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to purchase Bose QuietComfort 45. First search for items.', 'tool_calls': [{'id': 'fc_f86431cc-eee8-4bd7-a34c-6351bf9b9d5d', 'function': {'arguments': '{"query":"Bose QuietComfort 45"}', 'name': 'search_items'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 217, 'total_tokens': 265, 'completion_time': 0.10735181, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.008752131, 'prompt_tokens_details': None, 'queue_time': 0.051258188, 'total_time': 0.116103941}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d81b3304b3', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dab86-cab6-7861-b0a4